# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya

This notebook provides a step-by-step guide for exploring and processing the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. The dataset describes ordered logistic regression results for adoption predictors in rangeland management, and is authored by Kamadi, V, Chimoita, EL, Wahome, RG, and Odhong, C.

### Dataset Source
The dataset is described by a [Croissant schema JSON-LD file](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print high-level information
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview

Review available record sets and their IDs. List all fields for each record set, with their `@id`s, `name`, and `data_type` when available. This will help determine what tables and attributes are available for analysis.

In [ ]:
# List all record sets in the metadata.
record_sets = list(dataset.record_sets)
print(f"Total record sets in this dataset: {len(record_sets)}")

for rs in record_sets:
    print(f"\nRecord Set ID: {rs.id}")
    print(f"Name: {rs.name}")
    fields = rs.fields
    print(f"Fields ({len(fields)}):")
    for fld in fields:
        fld_line = f"  - @id: {fld.id} | name: {fld.name}"
        if hasattr(fld, 'data_type'):
            fld_line += f" | data_type: {fld.data_type}"
        print(fld_line)

## 3. Data Extraction

Load data from selected record sets into Pandas DataFrames for analysis. The `@id` values will be used to select which record set(s) to extract.

In [ ]:
# Choose record sets for extraction by their @id
all_record_set_ids = [rset.id for rset in record_sets]
print("Record set @ids available:")
for i, rset_id in enumerate(all_record_set_ids):
    print(f"  [{i}] {rset_id}")

# For demonstration, extract the first record set (replace the index to select others as needed)
selected_record_set_id = all_record_set_ids[0] if all_record_set_ids else None

if selected_record_set_id is not None:
    # Extract all data from the chosen record set
    records = list(dataset.records(record_set=selected_record_set_id))
    df = pd.DataFrame(records)
    print(f"\nFields (columns) in record set '{selected_record_set_id}':")
    print(df.columns.tolist())
    print("\nSample records:")
    display(df.head())
else:
    print('No record sets found in the dataset.')

## 4. Exploratory Data Analysis (EDA)

Here, we demonstrate basic EDA steps using numeric and categorical fields. We'll:

- Filter based on a numeric field value.
- Normalize the numeric field.
- Group by a categorical field.

Replace `<numeric_field_id>` and `<categorical_field_id>` below with actual `@id`s based on the printed field overview. 

In [ ]:
# Example: Replace these IDs with real field @ids from the previous cell
numeric_field = None  # e.g., 'http://mlcommons.org/croissant/dataset/logLikField' or similar
categorical_field = None  # e.g., 'http://mlcommons.org/croissant/dataset/wardField' or similar

available_fields = df.columns.tolist() if 'df' in locals() else []
print("Available fields:", available_fields)

# Try to autodetect a numeric field if user hasn't supplied one
import numpy as np
autonum = None
autocat = None
for col in available_fields:
    if np.issubdtype(df[col].dtype, np.number):
        autonum = col
        break
for col in available_fields:
    if df[col].dtype == object:
        autocat = col
        break

if numeric_field is None:
    numeric_field = autonum
if categorical_field is None:
    categorical_field = autocat

if numeric_field and numeric_field in df.columns:
    threshold = df[numeric_field].mean() if not pd.isna(df[numeric_field].mean()) else 0
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records where {numeric_field} > {threshold:.2f}:")
    print(filtered_df[[numeric_field]].head())

    # Normalize
    mean_val = filtered_df[numeric_field].mean()
    std_val = filtered_df[numeric_field].std() or 1
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - mean_val) / std_val
    print(f"\nNormalized {numeric_field} (first 5 rows):")
    print(filtered_df[[numeric_field, norm_col]].head())

    # Group by a categorical field, if one found
    if categorical_field and categorical_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(categorical_field)[numeric_field].agg(['count','mean','std']).reset_index()
        print(f"\nGrouped by '{categorical_field}':")
        print(grouped_df.head())
else:
    print('Could not find a numeric field for EDA. Please check the dataset structure.')

## 5. Visualization

Let's illustrate the distribution of the main numeric field and the group-wise means (if applicable).

Feel free to change the fields and customize the plots based on your analysis needs.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field and numeric_field in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

    if categorical_field and categorical_field in df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=df[categorical_field], y=df[numeric_field])
        plt.xticks(rotation=45, ha='right')
        plt.title(f"{numeric_field} by {categorical_field}")
        plt.tight_layout()
        plt.show()
else:
    print('Cannot plot; no suitable numeric field found.')

## 6. Conclusion

- Explored the FAIR^2 dataset's structure via the Croissant schema using `mlcroissant`.
- Loaded record sets and fields using official croissant `@id` references to maintain transparency and reproducibility.
- Demonstrated basic EDA, grouping, normalization, and data visualization.
- For deeper insights, examine regression outputs or demographic predictors further using the available record sets and fields.